# Text Classification on Indonesian Sentiment Data
## BINUS Text Mining — Session 9 · Live walkthrough

This notebook accompanies the guest lecture *Text Classification: Building, Evaluating & Explaining Models*.

**Dataset:** SmSA from [IndoNLU](https://huggingface.co/datasets/indonlp/indonlu) — Indonesian sentence-level sentiment (positive / neutral / negative).

**Learning outcomes covered**

- **C2 (Comprehension):** explain what each pipeline component does and why CV matters.
- **C3 (Application):** run the cells, modify the pipeline, try a different classifier.
- **C4 (Analysis):** read the coefficients, interpret the confusion matrix, explain a single prediction with LIME.

**One-time install** (uncomment if running for the first time):

In [ ]:
# !pip install --quiet scikit-learn sastrawi lime matplotlib pandas seaborn

---
## 1 · Load the dataset

We pull the SmSA sentiment dataset directly from the **IndoNLU GitHub repo** as TSV files. This avoids the `datasets` library entirely (which has deprecated script-based loaders), and gives us full control over the data.

Each TSV row is `<text>\t<label>` with no header. Labels are `positive`, `neutral`, `negative`.

In [ ]:
import pandas as pd

BASE = ("https://raw.githubusercontent.com/IndoNLP/indonlu/"
        "master/dataset/smsa_doc-sentiment-prosa")

def load_smsa(split: str) -> pd.DataFrame:
    """split ∈ {'train', 'valid', 'test'}"""
    url = f"{BASE}/{split}_preprocess.tsv"
    df = pd.read_csv(url, sep="\t", header=None,
                     names=["text", "label"], quoting=3, on_bad_lines="skip")
    return df.dropna().reset_index(drop=True)

df_train = load_smsa("train")
df_val   = load_smsa("valid")
df_test  = load_smsa("test")

label_names = sorted(df_train["label"].unique().tolist())
print("Sizes :", len(df_train), len(df_val), len(df_test))
print("Labels:", label_names)
df_train.head()

**Offline / corporate-firewall fallback.** If the raw.githubusercontent.com URLs are blocked on your network, download the three `.tsv` files manually from <https://github.com/IndoNLP/indonlu/tree/master/dataset/smsa_doc-sentiment-prosa>, place them next to this notebook, and replace the `BASE` string above with `BASE = '.'` (or wherever you saved the files).

**Alternative:** if you prefer the Hugging Face copy, you can use `load_dataset("indonlp/indonlu", "smsa", trust_remote_code=True)` — but only with `datasets<3.0`. From `datasets>=3.0`, script-based loaders were removed entirely, which is the root cause of the `Dataset scripts are no longer supported, but found indonlu.py` error. Loading the TSV directly (above) sidesteps the issue.

In [ ]:
# Class balance
df_train["label"].value_counts(normalize=True).round(3) * 100

**Observation:** the dataset is imbalanced — `positive` reviews dominate. This is exactly why we'll use macro-F1 instead of accuracy.

Let's look at a few representative rows in each class:

In [ ]:
for lbl in label_names:
    print(f"\n=== {lbl.upper()} ===")
    samples = df_train[df_train["label"] == lbl]["text"].head(3).tolist()
    for s in samples:
        print(" •>", s[:140])

---
## 2 · Indonesian preprocessing with Sastrawi

The biggest difference between English and Indonesian text classification is **affixation**. The single root `beri` (give) appears as `memberikan`, `diberikan`, `pemberian`, `pemberi`, `memberi`, etc. Without stemming, our vocabulary explodes and each variant gets its own feature.

[Sastrawi](https://github.com/har07/PySastrawi) is the de-facto Indonesian stemmer.

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from IPython.display import HTML, display
import re

stemmer = StemmerFactory().create_stemmer()

def clean_id(text: str) -> str:
    """Lower-case, strip URLs/mentions, then stem with Sastrawi."""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+|@\w+", " ", text)   # URLs / mentions
    text = re.sub(r"[^a-z\s]", " ", text)                  # keep letters
    text = re.sub(r"\s+", " ", text).strip()
    return stemmer.stem(text)

# For lecture demo purposes
# === same stemmer, but produces a side-by-side diff for the slide. ===
def show_stemming(text: str) -> None:
    """Display the original next to clean_id(text), with stemmed words bolded."""
    cleaned = clean_id(text)                  # ← reuses the original function

    # Word-by-word alignment for the highlight only (pipeline still uses clean_id)
    parts = []
    for tok in text.split():
        m = re.match(r"^(\W*)(.*?)(\W*)$", tok, flags=re.UNICODE)
        pre, word, post = m.groups()
        stem = stemmer.stem(word.lower())
        if stem and stem != word.lower():
            parts.append(f"{pre}<b style='color:#F77F00'>{stem}</b>"
                         f"<span style='color:#888;font-size:90%'> ({word})</span>{post}")
        else:
            parts.append(tok)
    highlighted = " ".join(parts)

    display(HTML(
        f"<div style='font-family:sans-serif;line-height:1.7;margin-bottom:1em'>"
        f"<b style='color:#1E3A5F'>Original          :</b> {text}<br>"
        f"<b style='color:#1E3A5F'>clean_id(text)    :</b> <code>{cleaned}</code><br>"
        f"<b style='color:#1E3A5F'>Word-by-word view :</b> {highlighted}"
        f"</div>"
    ))
    
# Try it on a sample review
sample = df_train.loc[0, "text"]
#print("BEFORE:", sample)
#print("")
#print("AFTER :", clean_id(sample))

# For lecture demo purposes
show_stemming(sample)

---
## 3 · Build the classification pipeline

Three primitives — `Pipeline`, `TfidfVectorizer`, `LogisticRegression`. Notice we pass our `clean_id` function as the `preprocessor` argument; the vectoriser will call it on every document.

Why **bigrams** (`ngram_range=(1, 2)`)?  Indonesian negation: `tidak rusak` (not broken) vs `tidak bagus` (not good). Single tokens can't capture this.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# for live demo - downsampling. 
# In a real run for the whole dataset, you may want to comment out the next two lines
df_train = df_train.sample(2000, random_state=42).reset_index(drop=True)
df_val   = df_val.sample(500,  random_state=42).reset_index(drop=True)

X_train, y_train = df_train["text"].tolist(), df_train["label"].tolist()
X_val,   y_val   = df_val["text"].tolist(),   df_val["label"].tolist()

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        preprocessor=clean_id,
        ngram_range=(1, 2),
        min_df=3, max_df=0.95,
    )),
    ("clf", LogisticRegression(
        max_iter=1000, C=1.0,
        class_weight="balanced",
        n_jobs=-1,
    )),
])

pipe.fit(X_train, y_train)
print("Validation accuracy :", round(pipe.score(X_val, y_val), 3))

### Look at the confusion matrix and per-class metrics

Don't stop at the accuracy number — different classes can have very different performance.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = pipe.predict(X_val)

cm = confusion_matrix(y_val, y_pred, labels=label_names)
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Validation confusion matrix — baseline pipeline")
plt.tight_layout(); plt.show()

print(classification_report(y_val, y_pred, digits=3))

**Read the matrix**, not just the accuracy: the model probably confuses `neutral` with `positive`. We'll look at *why* in the feature-importance section.

---
## 4 · Cross-Validation — getting a confidence interval, not just a number

A single train-test split is one sample from a distribution. We want to estimate the *mean* and *spread* of that distribution. **Stratified K-fold CV** does this without burning more data.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_val_score(
    pipe, X_train, y_train,
    cv=cv, scoring="f1_macro", n_jobs=-1,
)

print(f"Macro-F1 = {scores.mean():.3f} ± {scores.std():.3f}")
print("Per-fold:", scores.round(3).tolist())

Why **macro-F1** instead of accuracy? Because the dataset is imbalanced — accuracy gets dominated by the `positive` class. Macro-F1 averages F1 equally across classes, so a model that ignores `neutral` will be penalised.

---
## 5 · Feature importance — which Bahasa words drive the prediction?

Logistic regression coefficients are directly interpretable. For each class, every word has a weight; positive weight pushes toward the class, negative pulls away.

Pull the fitted vectoriser and classifier out of the pipeline with `pipe.named_steps[...]`.

In [ ]:
import numpy as np

vec = pipe.named_steps["tfidf"]
clf = pipe.named_steps["clf"]
vocab = np.array(vec.get_feature_names_out())

def top_features(class_label, n=15):
    idx = list(clf.classes_).index(class_label)
    coefs = clf.coef_[idx]
    top = np.argsort(coefs)[-n:][::-1]
    bot = np.argsort(coefs)[:n]
    return (
        pd.DataFrame({"word": vocab[top], "coef": coefs[top]}),
        pd.DataFrame({"word": vocab[bot], "coef": coefs[bot]}),
    )

for cls in label_names:
    print(f"\n=== TOP WORDS FOR '{cls}' ===")
    pos, neg = top_features(cls, n=10)
    print("Pushing TOWARD:\n", pos.to_string(index=False))
    print("Pushing AWAY:\n",   neg.to_string(index=False))

In [ ]:
# Visualise top words for the negative class
neg_top, _ = top_features("negative", n=15)
fig, ax = plt.subplots(figsize=(8, 5))
neg_top.set_index("word")["coef"][::-1].plot.barh(color="#D62828", ax=ax)
ax.set_title('Top 15 words pushing toward "negative"')
ax.set_xlabel("Logistic regression coefficient")
plt.tight_layout(); plt.show()

**Sanity check moment:** scan the top words. Do they look like an Indonesian human's mental model of negative reviews? If `Tokopedia` or another brand name appears with a strong weight, you have a *spurious correlation*, not a sentiment model. That's a bug — fix it by adding to a stopword list or rebalancing data across platforms.

---
## 6 · Hyperparameter tuning with GridSearchCV

Tune the **whole pipeline** in one call. Keys use the `step__param` syntax to reach into nested components.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

param_grid = {
    "tfidf__min_df":      [1, 3, 5],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__C":             [0.1, 1.0, 10.0],
}

gs = GridSearchCV(
    pipe, param_grid,
    cv=5, scoring="f1_macro",
    n_jobs=-1, verbose=1,
    return_train_score=True,
)
gs.fit(X_train, y_train)

print("Best params :", gs.best_params_)
print("Best CV F1  :", round(gs.best_score_, 3))
print("Test F1     :", round(
    f1_score(y_val, gs.predict(X_val), average="macro"), 3
))

**Key principle:** the test set is touched ONCE — at the end, after best params have been chosen from CV folds. Anything else is silent overfitting.

---
## 7 · Explain a single prediction with LIME

[LIME](https://github.com/marcotcr/lime) (Local Interpretable Model-agnostic Explanations) perturbs the input and fits a tiny linear surrogate model around it. Output: a contribution score per word.

Anchors (last week) gives you a *rule*. LIME gives you continuous *attributions*. They're complementary.

In [ ]:
from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=label_names)

review = ("Barang rusak ketika sampai, packaging bagus tapi isinya "
          "kecewa sekali. Tidak sesuai deskripsi.")

best_pipe = gs.best_estimator_
exp = explainer.explain_instance(
    review,
    classifier_fn=best_pipe.predict_proba,
    num_features=8,
    labels=[list(best_pipe.classes_).index("negative")],
)

# Show contributions as a table
print(f"Review: {review}\n")
print(f"Predicted: {best_pipe.predict([review])[0]}")
print("Class probabilities:",
      dict(zip(best_pipe.classes_, best_pipe.predict_proba([review])[0].round(3))))
print("\nLIME attributions toward 'negative':")
for word, weight in exp.as_list(label=list(best_pipe.classes_).index("negative")):
    sign = "⊕" if weight > 0 else "⊖"
    print(f"  {sign} {word:<20} {weight:+.3f}")

In [ ]:
# Pretty visual
exp.show_in_notebook(text=True, labels=[list(best_pipe.classes_).index("negative")])

---
## 8 · Try-it-yourself / reflection exercises

Pick at least two and submit your answers via the course Mentimeter / LMS.

1. **(C2)** Re-run cell 4 with `random_state=1, 2, 3` in `StratifiedKFold` and report mean ± std of macro-F1 across runs. Why does CV reduce reporting error?
2. **(C3)** Replace `LogisticRegression` with `LinearSVC`. Does macro-F1 go up or down? What do you lose in interpretability?
3. **(C3)** Add a custom Indonesian stopword list (try `Sastrawi.StopWordRemover.StopWordRemoverFactory`) and pass it into `TfidfVectorizer(stop_words=...)`. Effect on F1 and on the top-words list?
4. **(C4)** Run LIME on a *misclassified* example from the validation set. Which word(s) are most responsible for the wrong prediction?
5. **(C4)** Look at the `negative` class top-words list. Is any of them a *brand* or *platform* name (e.g., 'Tokopedia', 'Shopee', 'JNE')? If yes, what does that tell you about your data, and how would you fix it?

---
*Notebook © BINUS guest lecture · MIT licence on the code · Dataset: IndoNLU SmSA (MIT licence)*